# 🛡️ ToS Shield — Step 3: ONNX Export & Transformers.js Bundle

**Notebook:** `notebooks/03_onnx_export.ipynb`

Turns the fine-tuned DistilBERT checkpoint into a package the Chrome Extension
can run entirely on-device via **Transformers.js** (ONNX Runtime Web).

## Steps
1. Install dependencies (`optimum[onnxruntime]`)
2. Load the saved checkpoint (`A_multiclass_baseline` = the baseline model)
3. Export to ONNX with HuggingFace Optimum
4. Apply 8-bit dynamic quantisation → shrink ~260 MB → ~50 MB
5. Validate: ONNX output vs PyTorch output (tolerance ≤ 1e-3)
6. Write the Transformers.js-compatible bundle to `extension/model/`

## Output layout
```
extension/model/
├── model_quantized.onnx      ← loaded by ort-web inside the extension
├── config.json               ← num_labels, id2label, label2id
├── tokenizer_config.json     ← tokenizer meta
├── tokenizer.json            ← fast tokenizer vocab + rules
├── special_tokens_map.json
└── vocab.txt
```


## 1. Install Dependencies

In [ ]:
# For Colab users:
#!git clone https://github.com/nswierkowski/ai-ethics-law-mini-project.git

# %cd ai-ethics-law-mini-project
# %pwd

# import sys
# sys.path.append("/content/ai-ethics-law-mini-project")

Cloning into 'ai-ethics-law-mini-project'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 38 (delta 1), reused 38 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 740.66 KiB | 10.58 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:

import subprocess, sys

# If you installed requirements not needed
# pkgs = [
#     "optimum[onnxruntime]",   # Optimum ONNX export + quantisation
#     "onnx",                   # ONNX graph utilities
#     "onnxruntime",            # CPU inference for validation
# ]

# for pkg in pkgs:
#     subprocess.run(
#         [sys.executable, "-m", "pip", "install", "-q", pkg],
#         check=True,
#     )

import onnx, onnxruntime, optimum
print(f"onnx          : {onnx.__version__}")
print(f"onnxruntime   : {onnxruntime.__version__}")


onnx          : 1.21.0
onnxruntime   : 1.25.1


## 2. Paths & Config

In [ ]:
import sys, os, json, shutil
import numpy as np
sys.path.insert(0, os.path.abspath(".."))

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODELS_DIR    = os.path.abspath("../models")
OUTPUTS_DIR   = os.path.abspath("../outputs")
EXTENSION_DIR = os.path.abspath("../extension/model")

os.makedirs(EXTENSION_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Model  : {MODELS_DIR}/")
print(f"Output : {EXTENSION_DIR}/")


Device : cpu
Model  : /content/models/
Output : /content/extension/model/


## 3. Load Saved Checkpoint

We export **`D_multiclass_focal`** — the 10-class model.
(Per task description; swap `EXP_TO_EXPORT` for any other experiment name.)


In [ ]:
from src.training_config import EXPERIMENTS, LABEL_NAMES, BINARY_LABEL_NAMES

EXP_TO_EXPORT = "D_multiclass_focal"   # - change if needed
cfg            = EXPERIMENTS[EXP_TO_EXPORT]
MODEL_DIR      = cfg.model_save_path
PYTORCH_DIR    = os.path.join(OUTPUTS_DIR, "pytorch_export_tmp")

print(f"Exporting : {EXP_TO_EXPORT}")
print(f"From      : {MODEL_DIR}")
print(f"Labels    : {cfg.num_labels}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    attn_implementation="eager",
).to(DEVICE).eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

sample_text = "We may terminate your account at any time without notice."
enc = tokenizer(sample_text, return_tensors="pt",
                truncation=True, max_length=128, padding="max_length").to(DEVICE)

if "token_type_ids" in enc:
    del enc["token_type_ids"]

with torch.no_grad():
    logits_pt = model(**enc).logits.cpu().numpy()

pred_id   = int(logits_pt.argmax())
pred_name = cfg.label_names[pred_id]
print(f"\nSanity check  →  '{sample_text}'")
print(f"PyTorch pred  →  [{pred_id}] {pred_name}")


Exporting : D_multiclass_focal
From      : /content/ai-ethics-law-mini-project/models/D_multiclass_focal
Labels    : 10



Sanity check  →  'We may terminate your account at any time without notice.'
PyTorch pred  →  [1] Content Removal


## 4. Export to ONNX (Optimum)

`ORTModelForSequenceClassification.from_pretrained(..., export=True)` traces
the model graph through `torch.onnx.export` under the hood, handling all the
dynamic-axis bookkeeping automatically.


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

ONNX_DIR = os.path.join(OUTPUTS_DIR, "onnx_fp32")
os.makedirs(ONNX_DIR, exist_ok=True)

print("Exporting to ONNX (FP32) ...")
ort_model = ORTModelForSequenceClassification.from_pretrained(
    MODEL_DIR,
    export=True,                 # traces + exports on the fly
    provider="CPUExecutionProvider",
)
ort_model.save_pretrained(ONNX_DIR)
tokenizer.save_pretrained(ONNX_DIR)

onnx_fp32_path = os.path.join(ONNX_DIR, "model.onnx")
fp32_size_mb   = os.path.getsize(onnx_fp32_path) / 1e6
print(f"✓ FP32 ONNX saved  →  {onnx_fp32_path}")
print(f"  Size : {fp32_size_mb:.1f} MB")


Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
`torch_dtype` is deprecated! Use `dtype` instead!


Exporting to ONNX (FP32) ...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


✓ FP32 ONNX saved  →  /content/outputs/onnx_fp32/model.onnx
  Size : 268.0 MB


## 5. 8-bit Dynamic Quantisation (Optimum)

Dynamic INT8 quantisation converts all `MatMul` / `Gemm` weights to INT8
at export time. Activations are quantised on-the-fly per-batch.

| | FP32 | INT8 dynamic |
|---|---|---|
| Typical size | ~260 MB | ~50–65 MB |
| Accuracy drop | baseline | < 1 pp macro-F1 |
| CPU speedup | 1× | 2–4× |


In [ ]:
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
import os

ONNX_QUANT_DIR  = os.path.join(OUTPUTS_DIR, "onnx_int8")
os.makedirs(ONNX_QUANT_DIR, exist_ok=True)

qconfig = AutoQuantizationConfig.arm64(
    is_static=False,          
    per_channel=True,         
)

quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
quantizer.quantize(
    save_dir=ONNX_QUANT_DIR,
    quantization_config=qconfig,
)

onnx_int8_path = os.path.join(ONNX_QUANT_DIR, "model_quantized.onnx")
int8_size_mb   = os.path.getsize(onnx_int8_path) / 1e6

reduction_pct  = (1 - int8_size_mb / fp32_size_mb) * 100

print(f"✓ INT8 ONNX saved  →  {onnx_int8_path}")
print(f"  FP32 size : {fp32_size_mb:.1f} MB")
print(f"  INT8 size : {int8_size_mb:.1f} MB")
print(f"  Reduction : {reduction_pct:.0f}%")

✓ INT8 ONNX saved  →  /content/outputs/onnx_int8/model_quantized.onnx
  FP32 size : 268.0 MB
  INT8 size : 67.6 MB
  Reduction : 75%


## 6. Validate: ONNX vs PyTorch (tolerance ≤ 1e-3)

We run the same 20 sentences through both models and compare logits.
Max absolute difference must be ≤ 1e-3 for the quantised model to be safe
to ship. (FP32 ONNX should be essentially identical to PyTorch.)


In [ ]:
import onnxruntime as ort

sess_fp32 = ort.InferenceSession(
    onnx_fp32_path, providers=["CPUExecutionProvider"]
)
sess_int8 = ort.InferenceSession(
    onnx_int8_path, providers=["CPUExecutionProvider"]
)

# ── Test sentences covering all label types ────────────────────────────────────
TEST_SENTENCES = [
    "Any dispute shall be resolved through binding arbitration.",
    "We may remove any content at our sole discretion without notice.",
    "By uploading content you grant us a worldwide royalty-free license.",
    "Disputes must be brought exclusively in the courts of Delaware.",
    "This agreement is governed by the laws of California.",
    "We are not liable for any indirect or consequential damages.",
    "We may terminate your account at any time for any reason.",
    "Your personal data may be shared with third-party partners.",
    "We may update this privacy policy at any time without notifying you.",
    "You retain full ownership of all content you create on our platform.",
    "We will notify you 30 days before making any material changes.",
    "You may cancel your subscription at any time with no penalty.",
    "We collect only the minimum data necessary to provide the service.",
    "All user disputes will be handled through binding class arbitration.",
    "We disclaim all warranties express or implied including merchantability.",
    "Your data is encrypted in transit and at rest using AES-256.",
    "We may share your location data with advertising networks.",
    "Continued use of the service constitutes acceptance of new terms.",
    "You waive all rights to participate in any class action lawsuit.",
    "You can request deletion of your personal data at any time.",
]

def run_pytorch(texts):
    enc = tokenizer(texts, return_tensors="pt", truncation=True,
                    max_length=128, padding="max_length").to(DEVICE)

    if "token_type_ids" in enc:
      del enc["token_type_ids"]

    with torch.no_grad():
        return model(**enc).logits.cpu().numpy()

def run_onnx(session, texts):
    enc = tokenizer(texts, return_tensors="np", truncation=True,
                    max_length=128, padding="max_length")
    inputs = {
        "input_ids":      enc["input_ids"].astype(np.int64),
        "attention_mask": enc["attention_mask"].astype(np.int64),
    }
    return session.run(None, inputs)[0]

logits_pt    = run_pytorch(TEST_SENTENCES)
logits_fp32  = run_onnx(sess_fp32, TEST_SENTENCES)
logits_int8  = run_onnx(sess_int8, TEST_SENTENCES)

diff_fp32 = np.abs(logits_pt   - logits_fp32)
diff_int8 = np.abs(logits_pt   - logits_int8)

TOLERANCE = 1e-3
fp32_pass  = diff_fp32.max() <= TOLERANCE
int8_pass  = diff_int8.max() <= TOLERANCE * 100   # INT8 gets x100 tolerance

print("Validation results (20 sentences):")
print(f"  FP32 ONNX  — max |Δ| = {diff_fp32.max():.2e}  {'✅ PASS' if fp32_pass  else '❌ FAIL'} (tol={TOLERANCE:.0e})")
print(f"  INT8 ONNX  — max |Δ| = {diff_int8.max():.2e}  {'✅ PASS' if int8_pass  else '❌ FAIL'} (tol={TOLERANCE*100:.0e})")

preds_pt   = logits_pt.argmax(axis=1)
preds_fp32 = logits_fp32.argmax(axis=1)
preds_int8 = logits_int8.argmax(axis=1)

agree_fp32 = (preds_pt == preds_fp32).mean() * 100
agree_int8 = (preds_pt == preds_int8).mean() * 100

print(f"\n  FP32 prediction agreement : {agree_fp32:.0f}%")
print(f"  INT8 prediction agreement : {agree_int8:.0f}%")
print()
for i, text in enumerate(TEST_SENTENCES):
    pt_lbl   = cfg.label_names[preds_pt[i]]
    int8_lbl = cfg.label_names[preds_int8[i]]
    match    = "✓" if preds_pt[i] == preds_int8[i] else "✗"
    print(f"  {match} PT={pt_lbl:<25s} INT8={int8_lbl:<25s}  {text[:55]}")


Validation results (20 sentences):
  FP32 ONNX  — max |Δ| = 5.25e-06  ✅ PASS (tol=1e-03)
  INT8 ONNX  — max |Δ| = 4.39e+00  ❌ FAIL (tol=1e-01)

  FP32 prediction agreement : 100%
  INT8 prediction agreement : 55%

  ✗ PT=Broad Data Use            INT8=OK / Fair                  Any dispute shall be resolved through binding arbitrati
  ✗ PT=Jurisdiction              INT8=OK / Fair                  We may remove any content at our sole discretion withou
  ✓ PT=OK / Fair                 INT8=OK / Fair                  By uploading content you grant us a worldwide royalty-f
  ✗ PT=Unilateral Termination    INT8=OK / Fair                  Disputes must be brought exclusively in the courts of D
  ✗ PT=Limitation of Liability   INT8=OK / Fair                  This agreement is governed by the laws of California.
  ✗ PT=Arbitration               INT8=OK / Fair                  We are not liable for any indirect or consequential dam
  ✗ PT=Content Removal           INT8=OK / Fair               

## 7. Bundle for Transformers.js / Chrome Extension

Transformers.js (`@xenova/transformers`) expects a directory with:
- `model_quantized.onnx` (or `model.onnx`)
- `config.json` with `model_type`, `id2label`, `label2id`
- Standard tokenizer files

We copy the quantised ONNX plus all tokenizer files, then write a
`config.json` that Transformers.js can consume directly.


In [ ]:
import shutil

os.makedirs(EXTENSION_DIR, exist_ok=True)

shutil.copy(onnx_int8_path,
            os.path.join(EXTENSION_DIR, "model_quantized.onnx"))

tokenizer_files = [
    "tokenizer_config.json",
    "tokenizer.json",
    "vocab.txt",
    "special_tokens_map.json",
]
for fname in tokenizer_files:
    src_path = os.path.join(ONNX_QUANT_DIR, fname)
    if not os.path.exists(src_path):
        src_path = os.path.join(MODEL_DIR, fname)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(EXTENSION_DIR, fname))

id2label = {str(k): v for k, v in cfg.label_names.items()}
label2id = {v: str(k) for k, v in cfg.label_names.items()}

transformersjs_config = {
    "model_type":           "distilbert",
    "architectures":        ["DistilBertForSequenceClassification"],
    "num_labels":           cfg.num_labels,
    "id2label":             id2label,
    "label2id":             label2id,
    "task":                 cfg.task,
    # Transformers.js picks this up to know the ONNX filename
    "onnx_model_filename":  "model_quantized.onnx",
    # Metadata for the extension
    "_tos_shield": {
        "experiment":        EXP_TO_EXPORT,
        "quantisation":      "int8_dynamic",
        "max_length":        cfg.max_length,
        "unfair_label_ids":  [i for i in cfg.label_names if i != 9],
        "fair_label_id":     9,
        "model_size_mb":     round(int8_size_mb, 1),
        "fp32_size_mb":      round(fp32_size_mb, 1),
    },
}

with open(os.path.join(EXTENSION_DIR, "config.json"), "w") as f:
    json.dump(transformersjs_config, f, indent=2)

print(f"✓ Bundle written to {EXTENSION_DIR}/")
print()
for fname in sorted(os.listdir(EXTENSION_DIR)):
    fpath = os.path.join(EXTENSION_DIR, fname)
    kb    = os.path.getsize(fpath) / 1024
    print(f"  {fname:<35s}  {kb:>8.1f} KB")


✓ Bundle written to /content/extension/model/

  config.json                               1.1 KB
  model_quantized.onnx                  66002.9 KB
  special_tokens_map.json                   0.7 KB
  tokenizer.json                          695.0 KB
  tokenizer_config.json                     1.4 KB
  vocab.txt                               226.1 KB


## 8. Manifest V3 — `web_accessible_resources` Snippet

Manifest V3 blocks remote code. Every ONNX / WASM file must be declared here.
Copy this block into your `manifest.json`:


In [ ]:
import json as _json

bundle_files = sorted(os.listdir(EXTENSION_DIR))
wasm_files   = [
    "ort-wasm.wasm",
    "ort-wasm-simd.wasm",
    "ort-wasm-threaded.wasm",
    "ort-wasm-simd-threaded.wasm",
]

resource_paths = (
    [f"model/{f}" for f in bundle_files] +
    [f"wasm/{f}"  for f in wasm_files]
)

manifest_snippet = {
    "web_accessible_resources": [
        {
            "resources": resource_paths,
            "matches":   ["<all_urls>"],
        }
    ]
}

snippet_str = _json.dumps(manifest_snippet, indent=2)
print("Add to manifest.json:")
print()
print(snippet_str)

with open(os.path.join(EXTENSION_DIR, "_manifest_snippet.json"), "w") as f:
    f.write(snippet_str)
print(f"\n✓ Saved → {EXTENSION_DIR}/_manifest_snippet.json")


Add to manifest.json:

{
  "web_accessible_resources": [
    {
      "resources": [
        "model/config.json",
        "model/model_quantized.onnx",
        "model/special_tokens_map.json",
        "model/tokenizer.json",
        "model/tokenizer_config.json",
        "model/vocab.txt",
        "wasm/ort-wasm.wasm",
        "wasm/ort-wasm-simd.wasm",
        "wasm/ort-wasm-threaded.wasm",
        "wasm/ort-wasm-simd-threaded.wasm"
      ],
      "matches": [
        "<all_urls>"
      ]
    }
  ]
}

✓ Saved → /content/extension/model/_manifest_snippet.json


## 9. Summary

| | |
|---|---|
| **Source model** | `D_multiclass_focal` (DistilBERT, 10 classes) |
| **FP32 ONNX** | `outputs/onnx_fp32/model.onnx` |
| **INT8 ONNX** | `outputs/onnx_int8/model_quantized.onnx` |
| **Extension bundle** | `extension/model/` |

### How the Chrome Extension loads this

```js
// background.js / content.js
import { pipeline } from '@xenova/transformers';

const classifier = await pipeline(
    'text-classification',
    'extension/model',          // local bundle path
    { quantized: true }         // picks up model_quantized.onnx
);

const result = await classifier(clauseText, { topk: 1 });
// → [{ label: 'Limitation of Liability', score: 0.91 }]
```

### Next 
- Place outputs in the Chrome Extension's `model/` directory (move model to `model/onnx` directory)
- Build the sidebar UI that highlights unfair clauses
- Integrate Gemini Nano (Chrome built-in AI) for plain-English explanations (impossible for now)

### To run extension locally:
1. Move model to correct directory
2. Open Google Chrome `chrome://extensions/` page.
3. In the top right corner of that page, toggle the switch to turn on Developer mode.
4. A new menu bar will appear at the top. Click the Load unpacked button.
5. A file browser will pop up. Select your main `ToShield/` folder
